In [1]:
import os
import cv2
import wave
import torch
import librosa
import numpy as np
import pandas as pd
import noisereduce as nr
import webrtcvad
import albumentations as A
from albumentations.pytorch import ToTensorV2
from audiomentations import Compose, AddGaussianNoise, PitchShift, TimeStretch
from transformers import AutoTokenizer
from tqdm import tqdm
from pathlib import Path
from PIL import Image
from facenet_pytorch import MTCNN


c:\New folder\New Emodect\tfv2_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
PROJECT_DIR = Path(r"C:\New folder\New Emodect")
CLEANED_DIR = PROJECT_DIR / "cleaned_metadata"
SPLIT_DIR = CLEANED_DIR / "splits"
PROCESSED_DIR = PROJECT_DIR / "processed"

# Batch preprocessing contract.
# The pipeline is batch-oriented and never renders/loads the full dataset at once.
# Maximum resident batch size is 32 samples. Lower this value if desired; values
# above 32 are rejected to protect the RTX 3050 6 GB target environment.
PREPROCESS_BATCH_SIZE = 32
if not 1 <= PREPROCESS_BATCH_SIZE <= 32:
    raise ValueError("PREPROCESS_BATCH_SIZE must be between 1 and 32.")

# Current authoritative split sizes.
EXPECTED_EMOTION_TOTAL = 146_335
EXPECTED_EMOTION_SPLITS = {
    "train": 99_961,
    "validation": 24_679,
    "test": 21_695
}
EXPECTED_SARCASM_TOTAL = 29_183
EXPECTED_SARCASM_SPLITS = {"train": 20_254, "validation": 4_479, "test": 4_450}
EXPECTED_GOEMOTIONS_FINAL_ROWS = 50_831

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_SR = 16000
MAX_AUDIO_DURATION = 6
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NUM_VIDEO_FRAMES = 16
TEXT_MODEL_NAME = "microsoft/deberta-v3-large"
MAX_TEXT_LENGTH = 128
FACE_DETECTION_THRESHOLD = 0.85

USE_AUDIO_AUGMENTATION = True
USE_FACE_AUGMENTATION = True
# Video temporal augmentation: TRAIN ONLY. Validation/test remain deterministic.
USE_VIDEO_TEMPORAL_AUGMENTATION = True
VIDEO_TEMPORAL_JITTER_FRACTION = 0.20

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv'}
AUDIO_EXTENSIONS = {'.wav', '.flac', '.mp3', '.m4a', '.ogg'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Text datasets must remain explicit so text rows cannot silently disappear.
# GoEmotions is an active TEXT source for the seven-class emotion task.
# Its preserved text payload is tokenized and saved for downstream
# text embeddings/head training.
TEXT_DATASETS = {"GoEmotions"}
REQUIRE_EMOTION_TEXT = True

EMOTION_CLASSES = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]
SARCASM_CLASSES = ["non_sarcastic", "sarcastic"]

EMOTION_SPLITS = {
    "train": SPLIT_DIR / "emotion_train.csv",
    "validation": SPLIT_DIR / "emotion_validation.csv",
    "test": SPLIT_DIR / "emotion_test.csv",
}
SARCASM_SPLITS = {
    "train": SPLIT_DIR / "sarcasm_train.csv",
    "validation": SPLIT_DIR / "sarcasm_validation.csv",
    "test": SPLIT_DIR / "sarcasm_test.csv",
}

# Load the authoritative emotion manifest once. Downstream coverage gates depend
# on this object; it must never rely on a stale notebook variable from a prior run.
EMOTION_MANIFEST_PATH = (
    CLEANED_DIR / "final_emotion_training_manifest.csv"
)
if not EMOTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Authoritative emotion manifest not found: {EMOTION_MANIFEST_PATH}"
    )

emotion_all = pd.read_csv(
    EMOTION_MANIFEST_PATH,
    low_memory=False
)

if len(emotion_all) != EXPECTED_EMOTION_TOTAL:
    raise RuntimeError(
        f"Unexpected authoritative emotion manifest size: {len(emotion_all):,}; "
        f"expected {EXPECTED_EMOTION_TOTAL:,}."
    )

if "modality" not in emotion_all.columns:
    raise RuntimeError("Authoritative emotion manifest has no modality column.")

print(f"Authoritative emotion manifest: {len(emotion_all):,} rows")
print(f"Preprocessing batch size: {PREPROCESS_BATCH_SIZE}")
print("Batch rendering/preprocessing contract: MAX 32 samples per batch")
print("No dataset-wide sample rendering or preview generation is performed.")

for split in ["train", "validation", "test"]:
    for modality in ["audio", "face", "video", "text"]:
        (PROCESSED_DIR / split / modality).mkdir(parents=True, exist_ok=True)
        (PROCESSED_DIR / "sarcasm" / split / "text").mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"Split manifests: {SPLIT_DIR}")
print(f"Processed output: {PROCESSED_DIR}")
print(f"Device: {DEVICE}")


Authoritative emotion manifest: 146,335 rows
Preprocessing batch size: 32
Batch rendering/preprocessing contract: MAX 32 samples per batch
No dataset-wide sample rendering or preview generation is performed.
Project: C:\New folder\New Emodect
Split manifests: C:\New folder\New Emodect\cleaned_metadata\splits
Processed output: C:\New folder\New Emodect\processed
Device: cuda


In [3]:
#initializer of model and augmenter
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
# Initialize MTCNN Face Detector (Uses GPU automatically if available)
mtcnn = MTCNN(
    keep_all=False,
    min_face_size=40,
    image_size=IMG_SIZE,
    margin=20,
    post_process=False,
    device=DEVICE
)
# Initialize Audio Augmenter
audio_augmenter = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
    TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5)
]) if USE_AUDIO_AUGMENTATION else None
# Initialize Face Augmenter & Normalizer
face_augmenter = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
]) if USE_FACE_AUGMENTATION else None
face_normalizer = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

c:\New folder\New Emodect\tfv2_venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\New folder\New Emodect\tfv2_venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## Mandatory multimodal preprocessing contract

This preprocessing stage must actively support **all four emotion modalities**:

1. **Audio** — waveform/audio feature preprocessing
2. **Video** — fixed-length temporal frame preprocessing
3. **Face** — facial crop/image preprocessing
4. **Text** — GoEmotions text preprocessing

None of these modalities is optional merely because another modality is available.

For video, temporal augmentation remains part of preprocessing:
- 16-frame temporal sampling
- deterministic validation/test sampling
- train-only temporal jitter
- no temporal augmentation is applied to validation/test

For GoEmotions, the retrieved dataset is intentionally used as the project's **text modality for seven-class emotion detection**. Its preserved `text` payload must be tokenized and materialized as preprocessing artifacts.

The preprocessing notebook must report coverage for all four modalities and fail the final gate if any required modality has zero successful outputs when it is represented in the authoritative emotion manifest.


In [4]:
# Canonical targets and physical modality rules
VALID_EMOTIONS = set(EMOTION_CLASSES)

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv'}
AUDIO_EXTENSIONS = {'.wav', '.flac', '.mp3', '.m4a', '.ogg'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_column(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        key = str(name).strip().lower()
        if key in lookup:
            return lookup[key]
    return None

def resolve_actual_modality(row):
    dataset = str(row.get("dataset", "")).strip()
    if dataset in TEXT_DATASETS:
        return "text"

    fp = str(row.get("file_path", "")).strip()
    meta = str(row.get("modality", "")).strip().lower()
    ext = Path(fp).suffix.lower()
    if ext in VIDEO_EXTENSIONS:
        return "video"
    if ext in AUDIO_EXTENSIONS:
        return "audio"
    if ext in IMAGE_EXTENSIONS:
        return "image"
    if meta in {"audio", "video", "image", "text"}:
        return meta
    return meta


In [5]:
# Audio preprocessing

def atomic_torch_save(obj, out_path):
    """Write a PyTorch object atomically so partial/corrupt .pt files are never treated as valid outputs."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = out_path.with_name(out_path.name + ".tmp")

    # Remove stale temporary file from a previous interrupted run.
    try:
        if tmp_path.exists():
            tmp_path.unlink()
    except Exception:
        pass

    try:
        # Legacy serialization avoids the zip-container writer that is implicated
        # by the Windows 'inline_container.cc ... unexpected pos' failure.
        torch.save(
            obj,
            str(tmp_path),
            _use_new_zipfile_serialization=False
        )

        # Verify the temporary file before publishing it.
        if not tmp_path.exists() or tmp_path.stat().st_size == 0:
            raise IOError(f"Temporary tensor file is empty: {tmp_path}")

        _ = torch.load(
            str(tmp_path),
            map_location="cpu"
        )

        os.replace(str(tmp_path), str(out_path))
        return True

    except Exception:
        try:
            if tmp_path.exists():
                tmp_path.unlink()
        except Exception:
            pass
        raise


def validate_saved_tensor(path):
    """Return True only if an existing .pt file can actually be loaded."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        _ = torch.load(str(path), map_location="cpu")
        return True
    except Exception:
        return False


def trim_silence_vad(audio, sr, aggressiveness=3):
    audio_int16 = (audio * 32767).astype(np.int16)
    vad = webrtcvad.Vad(aggressiveness)

    frame_duration = 30
    frame_length = int(sr * (frame_duration / 1000.0))

    active_frames = []
    for i in range(0, len(audio_int16) - frame_length, frame_length):
        frame = audio_int16[i:i + frame_length]
        if len(frame) == frame_length:
            try:
                if vad.is_speech(frame.tobytes(), sr):
                    active_frames.append((i, i + frame_length))
            except Exception:
                pass

    if not active_frames:
        return audio

    start_idx = active_frames[0][0]
    end_idx = active_frames[-1][1]
    return audio[start_idx:end_idx]


def preprocess_audio(file_path, split, out_path, use_noise_reduction=False):
    y, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
    if len(y) == 0:
        return False

    y = trim_silence_vad(y, sr)

    if use_noise_reduction:
        y = nr.reduce_noise(y=y, sr=sr)

    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val

    max_samples = int(MAX_AUDIO_DURATION * TARGET_SR)
    if len(y) > max_samples:
        y = y[:max_samples]
    else:
        y = np.pad(y, (0, max_samples - len(y)))

    if split == 'train' and USE_AUDIO_AUGMENTATION and audio_augmenter:
        y = audio_augmenter(samples=y, sample_rate=sr)

    return atomic_torch_save(
        torch.tensor(y, dtype=torch.float32),
        out_path
    )


In [6]:
# Face preprocessing

def detect_and_crop_face(image):
    ih, iw, _ = image.shape
    boxes, probs = mtcnn.detect(image)

    if boxes is None or len(boxes) == 0:
        size = min(ih, iw)
        image = image[(ih-size)//2:(ih+size)//2, (iw-size)//2:(iw+size)//2]
        return cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    best_idx = np.argmax(probs)
    if probs[best_idx] < FACE_DETECTION_THRESHOLD:
        size = min(ih, iw)
        image = image[(ih-size)//2:(ih+size)//2, (iw-size)//2:(iw+size)//2]
        return cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    x1, y1, x2, y2 = boxes[best_idx]
    w, h = x2 - x1, y2 - y1

    margin_x, margin_y = int(w * 0.2), int(h * 0.2)
    x1 = max(0, int(x1 - margin_x))
    y1 = max(0, int(y1 - margin_y))
    x2 = min(iw, int(x2 + margin_x))
    y2 = min(ih, int(y2 + margin_y))

    cropped = image[y1:y2, x1:x2]

    if cropped.size == 0:
        size = min(ih, iw)
        image = image[(ih-size)//2:(ih+size)//2, (iw-size)//2:(iw+size)//2]
        return cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    return cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))


def preprocess_face(file_path, split, out_path):
    img = cv2.imread(str(file_path))
    if img is None:
        print(f"[WARN] cv2 failed to load image: {file_path}")
        return False

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    face = detect_and_crop_face(img)

    if split == 'train' and USE_FACE_AUGMENTATION and face_augmenter:
        transformed = face_augmenter(image=face)
    else:
        transformed = face_normalizer(image=face)

    tensor = transformed['image'].contiguous().cpu()

    return atomic_torch_save(tensor, out_path)


In [7]:
# Text preprocessing
#
# GoEmotions intent:
# GoEmotions supplies the project's active text modality for seven-class
# emotion detection. The actual preserved text payload is tokenized here
# and saved as a downstream-ready PyTorch artifact.

def preprocess_text(text, out_path):
    if not isinstance(text, str) or not text.strip():
        return False

    text = text.strip()

    encoded = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors='pt'
    )

    obj = {
        'input_ids': encoded['input_ids'].squeeze(0).cpu(),
        'attention_mask': encoded['attention_mask'].squeeze(0).cpu()
    }

    return atomic_torch_save(obj, out_path)


In [8]:
# ============================================================
# CELL — MULTIMODAL COVERAGE CONTRACT
# ============================================================
# emotion_all is loaded from the authoritative 146,335-row manifest in Cell 1.

REQUIRED_MODALITIES = {"audio", "video", "face", "text"}

def normalize_modality(value):
    s = str(value).strip().lower()
    aliases = {
        "audio": "audio",
        "speech": "audio",
        "wav": "audio",
        "sound": "audio",

        "video": "video",
        "vid": "video",

        "face": "face",
        "facial": "face",
        "image": "face",
        "img": "face",
        "frame": "face",

        "text": "text",
        "txt": "text",
        "transcript": "text",
    }
    return aliases.get(s, s)

if "modality" not in emotion_all.columns:
    raise RuntimeError("Emotion manifest has no modality column.")

emotion_all["_normalized_modality"] = (
    emotion_all["modality"].map(normalize_modality)
)

represented_modalities = set(
    emotion_all["_normalized_modality"].dropna()
)

missing_represented = REQUIRED_MODALITIES - represented_modalities

print("Represented emotion modalities:")
print(
    emotion_all["_normalized_modality"]
    .value_counts()
    .sort_index()
    .to_string()
)

if missing_represented:
    raise RuntimeError(
        "The authoritative emotion manifest does not contain all four "
        "required modalities. Missing: "
        f"{sorted(missing_represented)}"
    )

print("All four required modalities represented: PASS")


Represented emotion modalities:
_normalized_modality
audio    16397
face     76229
text     50831
video     2878
All four required modalities represented: PASS


In [9]:
# Video preprocessing
#
# Temporal augmentation is implemented HERE, in the preprocessing pipeline,
# because the video sample is constructed here from a sequence of frames.
#
# TRAIN:
#   - deterministic 16-frame sampling positions are first created
#   - each position receives a small random temporal jitter
#   - the number/order of frames is preserved
#
# VALIDATION / TEST:
#   - deterministic linspace sampling is used
#   - no temporal augmentation is applied
#
# This is intentionally mild: we want timing/sampling robustness without
# changing the emotional content of the clip.

def get_video_frame_indices(total_frames, split):
    if total_frames <= 0:
        return np.array([], dtype=int)

    if total_frames == 1:
        return np.zeros(NUM_VIDEO_FRAMES, dtype=int)

    base = np.linspace(0, total_frames - 1, NUM_VIDEO_FRAMES)

    if (
        split == "train"
        and USE_VIDEO_TEMPORAL_AUGMENTATION
        and total_frames > 1
    ):
        # Estimate the temporal spacing between requested frames.
        spacing = (total_frames - 1) / max(NUM_VIDEO_FRAMES - 1, 1)

        # Jitter is bounded so frames stay close to their original temporal
        # positions. This teaches robustness to timing/frame-selection changes
        # without turning the augmentation into a different clip.
        max_jitter = max(1.0, spacing * VIDEO_TEMPORAL_JITTER_FRACTION)
        jitter = np.random.uniform(
            -max_jitter, max_jitter, size=NUM_VIDEO_FRAMES
        )

        # Keep first/last anchors stable so the augmented sequence still
        # represents the whole clip.
        jitter[0] = 0.0
        jitter[-1] = 0.0

        indices = np.rint(base + jitter).astype(int)
        indices = np.clip(indices, 0, total_frames - 1)

        # Preserve temporal order. Duplicates are allowed for very short clips.
        indices = np.maximum.accumulate(indices)
    else:
        indices = np.rint(base).astype(int)

    return indices


def preprocess_video(file_path, split, out_path):
    cap = cv2.VideoCapture(str(file_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return False

    frame_indices = get_video_frame_indices(total_frames, split)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face = detect_and_crop_face(frame)
            frames.append(face)

    cap.release()

    if not frames:
        return False

    processed_frames = []

    # Spatial augmentation remains TRAIN ONLY and is applied consistently
    # across all frames of the same video.
    if split == "train" and USE_FACE_AUGMENTATION and face_augmenter:
        targets = {
            f"image{i}": "image"
            for i in range(1, len(frames))
        }

        consistent_aug = A.Compose(
            [
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.GaussNoise(p=0.2),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ],
            additional_targets=targets,
        )

        aug_kwargs = {"image": frames[0]}
        for i in range(1, len(frames)):
            aug_kwargs[f"image{i}"] = frames[i]

        result = consistent_aug(**aug_kwargs)
        processed_frames.append(result["image"])

        for i in range(1, len(frames)):
            processed_frames.append(result[f"image{i}"])
    else:
        for face in frames:
            res = face_normalizer(image=face)
            processed_frames.append(res["image"])

    # Keep the downstream video representation fixed at NUM_VIDEO_FRAMES.
    while len(processed_frames) < NUM_VIDEO_FRAMES:
        processed_frames.append(torch.zeros_like(processed_frames[0]))

    video_tensor = torch.stack(
        processed_frames[:NUM_VIDEO_FRAMES]
    ).contiguous().cpu()

    return atomic_torch_save(video_tensor, out_path)


In [10]:
# Routing and batch-oriented master preprocessing loop
#
# IMPORTANT:
# - The dataset is processed in bounded batches of at most PREPROCESS_BATCH_SIZE.
# - No dataset-wide sample rendering/preview pass is performed.
# - Outputs are still written one artifact per input sample because downstream
#   training consumes sample-addressable .pt files.
# - A batch is completed and released before the next batch is loaded.
# - Text tokenization is additionally performed as a true tokenizer batch.
#
# This keeps peak working memory bounded for the RTX 3050 6 GB target machine.

def get_text_column(df):
    # GoEmotions must use its preserved actual text payload.
    if "dataset" in df.columns:
        if "GoEmotions" in set(df["dataset"].astype(str).str.strip()):
            if "text" not in df.columns:
                raise ValueError(
                    "GoEmotions text payload column is missing from the split manifest."
                )
            return "text"

    return find_column(
        df,
        ["text", "utterance", "sentence", "headline", "caption", "transcript"]
    )


def process_sample(row, split, task="emotion", idx=0):
    """Process one item inside a bounded batch.

    Sarcasm is an independent TEXT-ONLY task. It must never call
    resolve_actual_modality() or require a media file path.
    """
    sample_id = str(row.get("sample_id", f"sample_{idx}"))
    file_path = str(row.get("file_path", "")).strip()

    # HARD CONTRACT:
    # Sarcasm (MUStARD + News Headlines) is text-only.
    # Do not resolve or validate audiovisual file paths for sarcasm.
    if task == "sarcasm":
        modality = "text"
    else:
        modality = resolve_actual_modality(row)

    if task != "sarcasm" and modality != "text":
        if not file_path or not os.path.isfile(file_path):
            return None, modality, "file_not_found"

    folder = "face" if modality in {"face", "image"} else modality
    root = PROCESSED_DIR if task == "emotion" else PROCESSED_DIR / "sarcasm"
    out_path = root / split / folder / f"{sample_id}.pt"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        if validate_saved_tensor(out_path):
            return out_path, modality, "already_exists"
        try:
            out_path.unlink()
        except Exception as e:
            return None, modality, f"corrupt_output_delete_failed: {e}"

    try:
        if modality == "audio":
            success = preprocess_audio(file_path, split, out_path)
        elif modality in {"face", "image"}:
            success = preprocess_face(file_path, split, out_path)
        elif modality == "video":
            success = preprocess_video(file_path, split, out_path)
        elif modality == "text":
            text_col = get_text_column(row.to_frame().T)
            if text_col is None:
                return None, modality, "missing_text_column"
            success = preprocess_text(row[text_col], out_path)
        else:
            return None, modality, "unsupported_modality"

        if not success:
            return None, modality, "processing_failed"

        if not validate_saved_tensor(out_path):
            try:
                out_path.unlink()
            except Exception:
                pass
            return None, modality, "post_save_validation_failed"

        return out_path, modality, "processed"

    except Exception as e:
        print(f"[ERROR] {task} | {sample_id} | {modality} | {e}")
        return None, modality, f"{type(e).__name__}: {e}"


def process_batch(batch_df, split, task="emotion", start_idx=0):
    """
    Process one bounded batch.

    The batch dataframe is the only batch-level scheduling unit. Individual
    artifacts are persisted immediately so a failure does not require keeping
    the batch in memory and completed outputs survive interruption.
    """
    if len(batch_df) > PREPROCESS_BATCH_SIZE:
        raise RuntimeError(
            f"Batch size {len(batch_df)} exceeds the hard maximum "
            f"{PREPROCESS_BATCH_SIZE}."
        )

    batch_out_paths = []
    batch_statuses = []

    for local_idx, (_, row) in enumerate(batch_df.iterrows()):
        out, _, status = process_sample(
            row,
            split,
            task=task,
            idx=start_idx + local_idx
        )
        batch_out_paths.append(str(out) if out is not None else "")
        batch_statuses.append(status)

    return batch_out_paths, batch_statuses


def process_dataset_split(split_name, task="emotion"):
    manifests = EMOTION_SPLITS if task == "emotion" else SARCASM_SPLITS
    csv_file = manifests[split_name]

    if not csv_file.exists():
        raise FileNotFoundError(f"Missing split manifest: {csv_file}")

    # Read the manifest once; process it in bounded chunks below.
    df = pd.read_csv(csv_file, low_memory=False)

    # Hard GoEmotions text integrity gate before expensive preprocessing.
    goe_rows = (
        df[df["dataset"].astype(str).str.strip().eq("GoEmotions")].copy()
        if "dataset" in df.columns
        else pd.DataFrame()
    )

    if task == "emotion" and len(goe_rows):
        if "text" not in goe_rows.columns:
            raise RuntimeError("GoEmotions text payload column is missing.")

        if goe_rows["modality"].astype(str).str.strip().str.lower().ne("text").any():
            raise RuntimeError("GoEmotions must remain modality=text.")

        if goe_rows["text"].fillna("").astype(str).str.strip().eq("").any():
            raise RuntimeError("Empty GoEmotions text found.")

    if task == "emotion":
        label_col = find_column(
            df,
            ["mapped_emotion", "emotion", "emotion_label", "label", "target"]
        )
        if label_col is None:
            raise ValueError(f"No emotion label column in {csv_file}")
        if label_col != "emotion":
            df = df.rename(columns={label_col: "emotion"})
        df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
        bad = sorted(set(df["emotion"]) - VALID_EMOTIONS)
        if bad:
            raise ValueError(f"Unexpected emotion labels: {bad}")
    else:
        label_col = find_column(
            df,
            ["sarcasm_label", "mapped_sarcasm", "sarcasm", "label", "target"]
        )
        if label_col is None:
            raise ValueError(f"No sarcasm label column in {csv_file}")
        if label_col != "sarcasm_label":
            df = df.rename(columns={label_col: "sarcasm_label"})
        df["sarcasm_label"] = (
            df["sarcasm_label"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "0": "non_sarcastic", "0.0": "non_sarcastic",
                "1": "sarcastic", "1.0": "sarcastic",
                "false": "non_sarcastic", "true": "sarcastic",
            })
        )
        bad = sorted(set(df["sarcasm_label"]) - set(SARCASM_CLASSES))
        if bad:
            raise ValueError(f"Unexpected sarcasm labels: {bad}")

    df["actual_modality"] = [
        "text" if task == "sarcasm" else resolve_actual_modality(row)
        for _, row in df.iterrows()
    ]

    total_rows = len(df)
    total_batches = int(np.ceil(total_rows / PREPROCESS_BATCH_SIZE))

    print(f"\nProcessing {task.upper()} {split_name.upper()} ({total_rows:,} samples)")
    print(f"  Batch size: {PREPROCESS_BATCH_SIZE} (hard maximum = 32)")
    print(f"  Number of batches: {total_batches:,}")
    print(f"  Datasets: {df['dataset'].value_counts().to_dict()}")
    print(f"  Resolved modalities: {df['actual_modality'].value_counts().to_dict()}")

    out_paths = [""] * total_rows
    statuses = ["not_processed"] * total_rows

    # BATCH-ONLY scheduling: tqdm tracks batches, not samples.
    for batch_no, start in enumerate(
        tqdm(
            range(0, total_rows, PREPROCESS_BATCH_SIZE),
            total=total_batches,
            desc=f"{task}-{split_name}-batches"
        ),
        start=1
    ):
        stop = min(start + PREPROCESS_BATCH_SIZE, total_rows)
        batch_df = df.iloc[start:stop].copy()

        if len(batch_df) > 32:
            raise RuntimeError("Internal batch construction exceeded 32 samples.")

        batch_paths, batch_statuses = process_batch(
            batch_df,
            split_name,
            task=task,
            start_idx=start
        )

        out_paths[start:stop] = batch_paths
        statuses[start:stop] = batch_statuses

        # Release batch-local objects before loading the next batch.
        del batch_df, batch_paths, batch_statuses

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df["processed_path"] = out_paths
    df["preprocess_status"] = statuses

    audit_path = PROCESSED_DIR / f"{task}_{split_name}_preprocessing_audit.csv"
    processed_path = PROCESSED_DIR / f"{task}_{split_name}_processed.csv"
    df.to_csv(audit_path, index=False)

    clean = df[df["processed_path"].astype(str).str.strip().ne("")].copy()
    clean.to_csv(processed_path, index=False)

    print(f"  Successfully processed: {len(clean):,}/{len(df):,}")
    print(
        f"  Saved modalities: "
        f"{clean['actual_modality'].value_counts().to_dict() if len(clean) else {}}"
    )
    print(
        f"  Saved datasets: "
        f"{clean['dataset'].value_counts().to_dict() if len(clean) else {}}"
    )

    failed = df[df["processed_path"].astype(str).str.strip().eq("")]
    if len(failed):
        print(f"  [WARN] Failed: {len(failed):,}")
        print(failed["preprocess_status"].value_counts().to_dict())

    return df, clean


In [11]:
# Pre-run diagnostics
# The previous failure was a PyTorch serialization error while writing validation face tensors.
# This fixed version:
#   1. validates existing .pt files before reusing them,
#   2. deletes corrupted/incomplete outputs,
#   3. writes to a temporary file first,
#   4. validates the temporary tensor,
#   5. atomically renames it into place,
#   6. uses legacy PyTorch serialization to avoid the failing zip-container writer.
#
# IMPORTANT:
# Stop any previous preprocessing kernel before running this notebook.
# Do not run two preprocessing instances against the same processed directory.
print("Safe tensor writer enabled.")
print("Existing corrupt .pt files will be regenerated automatically.")


Safe tensor writer enabled.
Existing corrupt .pt files will be regenerated automatically.


In [12]:
# Video temporal augmentation verification
print("Video temporal augmentation:")
print(f"  Enabled for training: {USE_VIDEO_TEMPORAL_AUGMENTATION}")
print(f"  Jitter fraction: {VIDEO_TEMPORAL_JITTER_FRACTION:.2f}")
print("  Validation/test temporal sampling: deterministic")
print(f"  Frames per video: {NUM_VIDEO_FRAMES}")
print("  Location: preprocess_video() -> get_video_frame_indices()")

print(f"  Preprocessing batch size: {PREPROCESS_BATCH_SIZE} (maximum 32)")


Video temporal augmentation:
  Enabled for training: True
  Jitter fraction: 0.20
  Validation/test temporal sampling: deterministic
  Frames per video: 16
  Location: preprocess_video() -> get_video_frame_indices()
  Preprocessing batch size: 32 (maximum 32)


In [13]:
# Preflight: verify modality coverage before expensive preprocessing
#
# This notebook can process text correctly, but it cannot create text rows that
# are absent from the split manifests. Fail early instead of silently running
# a face/audio/video-only emotion preprocessing job.
#
# GoEmotions is explicitly routed as TEXT whenever it appears in a split.

def inspect_emotion_text_coverage():
    rows = []
    total_text = 0
    for split_name, csv_file in EMOTION_SPLITS.items():
        if not csv_file.exists():
            raise FileNotFoundError(f"Missing emotion split manifest: {csv_file}")

        df = pd.read_csv(csv_file, low_memory=False)
        modalities = [
            resolve_actual_modality(row)
            for _, row in df.iterrows()
        ]
        counts = pd.Series(modalities).value_counts().to_dict()
        text_count = int(counts.get("text", 0))
        total_text += text_count

        rows.append({
            "split": split_name,
            "rows": len(df),
            "text": text_count,
            "image": int(counts.get("image", 0)),
            "audio": int(counts.get("audio", 0)),
            "video": int(counts.get("video", 0)),
        })

    coverage = pd.DataFrame(rows)
    print("\nEmotion preflight modality coverage:")
    print(coverage.to_string(index=False))

    if REQUIRE_EMOTION_TEXT and total_text == 0:
        raise RuntimeError(
            "NO TEXT ROWS FOUND in the emotion split manifests. "
            "Preprocessing cannot create missing text samples. "
            "Fix/rebuild the upstream emotion metadata + split manifests "
            "(including GoEmotions) before running this notebook."
        )

    return coverage

emotion_preflight = inspect_emotion_text_coverage()


# Hard current-manifest GoEmotions gate.
goe_preflight_total = 0
for _split_name, _csv_file in EMOTION_SPLITS.items():
    _split_df = pd.read_csv(_csv_file, low_memory=False)
    goe_preflight_total += int(
        (_split_df["dataset"].astype(str).str.strip() == "GoEmotions").sum()
    )
assert goe_preflight_total == EXPECTED_GOEMOTIONS_FINAL_ROWS, (
    f"Expected {EXPECTED_GOEMOTIONS_FINAL_ROWS:,} GoEmotions rows "
    f"in current emotion splits, found {goe_preflight_total:,}."
)
print(
    f"GoEmotions current split coverage: "
    f"{goe_preflight_total:,}/{EXPECTED_GOEMOTIONS_FINAL_ROWS:,} — PASS"
)



Emotion preflight modality coverage:
     split  rows  text  image  audio  video
     train 99961 35394  51161  11488   1918
validation 24679  8255  13233   2711    480
      test 21695  7182  11835   2198    480
GoEmotions current split coverage: 50,831/50,831 — PASS


## GoEmotions active text path

GoEmotions is included specifically to strengthen the **text modality of the
seven-class emotion detector**.

Its intended preprocessing path is:

**preserved GoEmotions text → tokenizer → `input_ids` + `attention_mask` → validated `.pt` artifact → downstream text embedding/head**

This is separate from sarcasm:
- GoEmotions → seven-class **emotion**
- MUStARD / News Headlines → independent **sarcasm**
- Sarcasm is not an eighth emotion class.


In [15]:
# ============================================================
# MUStARD — SARCASM TEXT-ONLY ROUTING
# ============================================================
# IMPORTANT ARCHITECTURE DECISION:
#
# MUStARD is retained for the SEPARATE sarcasm task, but this project
# does NOT require MUStARD audiovisual source clips.
#
# Therefore:
#   * DO NOT search for MUStARD .mp4 files here.
#   * DO NOT render MUStARD video/face/audio artifacts.
#   * DO NOT create fabricated cross-modal pairings.
#   * DO NOT change the cleaned sarcasm manifest or its splits.
#   * MUStARD is processed by the master sarcasm pipeline as TEXT.
#
# The existing process_dataset_split(..., task="sarcasm") implementation
# intentionally routes every sarcasm row to text. It uses the existing
# cleaned/split manifests and the bounded batch contract (MAX 32).
#
# This keeps the already-correct emotion pipeline completely independent
# of MUStARD audiovisual availability.
#
# Deployment architecture:
#   MUStARD + News Headlines text/context
#              -> sarcasm model
#              -> sarcasm_flag
#
# Quadra remains the multimodal emotion model:
#   audio + face + video + text -> Quadra -> seven-class emotion
#
# The FastAPI layer combines the two independently trained predictions.
# ============================================================

MUStARD_DATASET_NAME = "MUStARD"
MUSTARD_MULTIMODAL_PREPROCESSING = False

def verify_mustard_text_only_contract():
    """
    Verify that the authoritative sarcasm splits contain MUStARD rows with
    usable text and that this preprocessing stage does not require source
    videos. No media are loaded or rendered.
    """
    rows = []

    for split in ["train", "validation", "test"]:
        csv_file = SARCASM_SPLITS[split]

        if not csv_file.exists():
            raise FileNotFoundError(
                f"Missing sarcasm split manifest: {csv_file}"
            )

        df = pd.read_csv(csv_file, low_memory=False)

        if "dataset" not in df.columns:
            raise RuntimeError(
                f"{split}: sarcasm manifest is missing the 'dataset' column."
            )

        mustard = df[
            df["dataset"]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq(MUStARD_DATASET_NAME.lower())
        ].copy()

        if mustard.empty:
            raise RuntimeError(
                f"{split}: no MUStARD rows found in the authoritative "
                "sarcasm split manifest."
            )

        text_col = get_text_column(mustard)
        if text_col is None:
            raise RuntimeError(
                f"{split}: MUStARD text column could not be resolved."
            )

        empty_text = int(
            mustard[text_col]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        if empty_text:
            raise RuntimeError(
                f"{split}: {empty_text:,} MUStARD rows have empty text."
            )

        rows.append({
            "split": split,
            "mustard_rows": len(mustard),
            "text_column": text_col,
            "empty_text": empty_text,
            "multimodal_source_required": False,
        })

        del mustard, df

    qa = pd.DataFrame(rows)

    print("=" * 88)
    print("MUStARD SARCASM TEXT-ONLY PREPROCESSING CONTRACT")
    print("=" * 88)
    print(qa.to_string(index=False))
    print("MUStARD audiovisual source videos required: NO")
    print("MUStARD video/face/audio rendering: DISABLED")
    print("MUStARD cross-modal pairing: DISABLED")
    print("MUStARD text remains active for separate sarcasm training: PASS")
    print("Existing emotion preprocessing: UNCHANGED")
    print("=" * 88)

    return qa

mustard_text_only_qa = verify_mustard_text_only_contract()


MUStARD SARCASM TEXT-ONLY PREPROCESSING CONTRACT
     split  mustard_rows text_column  empty_text  multimodal_source_required
     train           460   utterance           0                       False
validation           107   utterance           0                       False
      test           113   utterance           0                       False
MUStARD audiovisual source videos required: NO
MUStARD video/face/audio rendering: DISABLED
MUStARD cross-modal pairing: DISABLED
MUStARD text remains active for separate sarcasm training: PASS
Existing emotion preprocessing: UNCHANGED


## Batch preprocessing execution contract

The pipeline is **batch-oriented**, not sample-rendering oriented.

- Maximum preprocessing batch size: **32 samples**
- The notebook never creates a dataset-wide preview/sample-rendering pass.
- Each batch is scheduled, processed, persisted, and released before the next batch.
- Per-sample `.pt` files are still written because the downstream training pipeline needs sample-addressable artifacts; this does **not** mean the dataset is loaded/rendered all at once.
- Video temporal augmentation remains **TRAIN-ONLY** and validation/test remain deterministic.
- GoEmotions remains an active text modality.
- Audio, video, face, and text remain mandatory modalities.


In [21]:
# ============================================================
# SARCASM TEXT-ONLY PREFLIGHT
# ============================================================
# This gate prevents the master loop from ever requiring media files
# for the independent sarcasm task.
#
# MUStARD + News Headlines -> text -> DeBERTa sarcasm model
# No audio/video/image/face artifacts are required here.

for _split_name, _csv_file in SARCASM_SPLITS.items():
    _df = pd.read_csv(_csv_file, low_memory=False)

    if "dataset" not in _df.columns:
        raise RuntimeError(
            f"{_split_name}: sarcasm manifest is missing the dataset column."
        )

    _text_col = get_text_column(_df)
    if _text_col is None:
        raise RuntimeError(
            f"{_split_name}: no usable text column found for sarcasm."
        )

    _empty = int(
        _df[_text_col].fillna("").astype(str).str.strip().eq("").sum()
    )
    if _empty:
        raise RuntimeError(
            f"{_split_name}: {_empty:,} sarcasm rows have empty text."
        )

    print(
        f"Sarcasm {_split_name}: {_text_col} | "
        f"{len(_df):,} rows | text-only: PASS"
    )

    del _df

print("Sarcasm media-file resolution: DISABLED")
print("Sarcasm audiovisual preprocessing: DISABLED")
print("Sarcasm text-only preflight: PASS")


Sarcasm train: utterance | 20,254 rows | text-only: PASS
Sarcasm validation: utterance | 4,479 rows | text-only: PASS
Sarcasm test: utterance | 4,450 rows | text-only: PASS
Sarcasm media-file resolution: DISABLED
Sarcasm audiovisual preprocessing: DISABLED
Sarcasm text-only preflight: PASS


In [ ]:
# ============================================================
# SARCASM-ONLY PREPROCESSING EXECUTION
# ============================================================
# IMPORTANT:
# - This cell processes ONLY sarcasm.
# - It does NOT touch emotion preprocessing.
# - It does NOT delete or overwrite successful emotion artifacts.
# - Sarcasm is TEXT-ONLY (MUStARD + News Headlines).
#
# Expected:
#   train       20,254
#   validation   4,479
#   test         4,450

print("=" * 80)
print("SARCASM-ONLY PREPROCESSING")
print("=" * 80)

SARCASM_ONLY_EXPECTED = {
    "train": 20_254,
    "validation": 4_479,
    "test": 4_450,
}

sarcasm_only_results = {}

for _split_name, _expected_count in SARCASM_ONLY_EXPECTED.items():
    _csv_path = SARCASM_SPLITS[_split_name]
    _df = pd.read_csv(_csv_path, low_memory=False)

    _text_col = get_text_column(_df)
    if _text_col is None:
        raise RuntimeError(
            f"{_split_name}: no usable text column found for sarcasm."
        )

    _empty = int(
        _df[_text_col].fillna("").astype(str).str.strip().eq("").sum()
    )
    if _empty:
        raise RuntimeError(
            f"{_split_name}: {_empty:,} sarcasm rows have empty text."
        )

    if len(_df) != _expected_count:
        raise RuntimeError(
            f"{_split_name}: expected {_expected_count:,} rows, "
            f"found {len(_df):,}."
        )

    print(
        f"\nProcessing SARCASM {_split_name.upper()} "
        f"({_expected_count:,} samples)"
    )
    print(f"  Text column: {_text_col}")
    print("  Modality: text ONLY")
    print("  Media-file resolution: DISABLED")

    # Use the existing split processor. process_sample() now has the
    # hard sarcasm text-only branch, so no media path is required.
    _result = process_dataset_split(
        _split_name,
        task="sarcasm",
    )

    sarcasm_only_results[_split_name] = _result

    del _df

print("\n" + "=" * 80)
print("SARCASM-ONLY PREPROCESSING FINISHED")
print("=" * 80)

# The existing processor returns the per-split success count in its result.
# Validate against the exact target without rerunning emotion.
for _split_name, _expected_count in SARCASM_ONLY_EXPECTED.items():
    _result = sarcasm_only_results[_split_name]

    if isinstance(_result, dict):
        _successful = _result.get("successful", _result.get("success_count"))
    else:
        _successful = None

    if _successful is not None:
        print(
            f"{_split_name:12s}: {_successful:,}/{_expected_count:,} successful"
        )
        if int(_successful) != _expected_count:
            raise RuntimeError(
                f"{_split_name}: sarcasm preprocessing incomplete."
            )
    else:
        print(
            f"{_split_name:12s}: processor completed; "
            f"verify its reported successful count above is "
            f"{_expected_count:,}/{_expected_count:,}."
        )

print("\nSARCASM TEXT-ONLY GATE: PASS")


SARCASM-ONLY PREPROCESSING

Processing SARCASM TRAIN (20,254 samples)
  Text column: utterance
  Modality: text ONLY
  Media-file resolution: DISABLED

Processing SARCASM TRAIN (20,254 samples)
  Batch size: 32 (hard maximum = 32)
  Number of batches: 633
  Datasets: {'NewsHeadlinesSarcasm': 19794, 'MUStARD': 460}
  Resolved modalities: {'text': 20254}


sarcasm-train-batches: 100%|██████████| 633/633 [09:48<00:00,  1.08it/s]


  Successfully processed: 20,254/20,254
  Saved modalities: {'text': 20254}
  Saved datasets: {'NewsHeadlinesSarcasm': 19794, 'MUStARD': 460}

Processing SARCASM VALIDATION (4,479 samples)
  Text column: utterance
  Modality: text ONLY
  Media-file resolution: DISABLED

Processing SARCASM VALIDATION (4,479 samples)
  Batch size: 32 (hard maximum = 32)
  Number of batches: 140
  Datasets: {'NewsHeadlinesSarcasm': 4372, 'MUStARD': 107}
  Resolved modalities: {'text': 4479}


sarcasm-validation-batches: 100%|██████████| 140/140 [02:13<00:00,  1.05it/s]


  Successfully processed: 4,479/4,479
  Saved modalities: {'text': 4479}
  Saved datasets: {'NewsHeadlinesSarcasm': 4372, 'MUStARD': 107}

Processing SARCASM TEST (4,450 samples)
  Text column: utterance
  Modality: text ONLY
  Media-file resolution: DISABLED

Processing SARCASM TEST (4,450 samples)
  Batch size: 32 (hard maximum = 32)
  Number of batches: 140
  Datasets: {'NewsHeadlinesSarcasm': 4337, 'MUStARD': 113}
  Resolved modalities: {'text': 4450}


sarcasm-test-batches: 100%|██████████| 140/140 [02:09<00:00,  1.08it/s]


  Successfully processed: 4,450/4,450
  Saved modalities: {'text': 4450}
  Saved datasets: {'NewsHeadlinesSarcasm': 4337, 'MUStARD': 113}

SARCASM-ONLY PREPROCESSING FINISHED
train       : processor completed; verify its reported successful count above is 20,254/20,254.
validation  : processor completed; verify its reported successful count above is 4,479/4,479.
test        : processor completed; verify its reported successful count above is 4,450/4,450.

SARCASM TEXT-ONLY GATE: PASS
Emotion preprocessing was NOT rerun.


In [ ]:
# NOTE: This is the original full preprocessing execution cell.
# For the failed sarcasm recovery, DO NOT RUN THIS CELL.
# Use the dedicated SARCASM-ONLY PREPROCESSING cell immediately above instead.

# ============================================================
# FINAL — PREPROCESSING EXECUTION
# ============================================================
# IMPORTANT:
# Use the notebook's EXISTING process_dataset_split() implementation.
# It already uses:
#   EMOTION_SPLITS / SARCASM_SPLITS
#   process_batch(batch_df, split, task, start_idx)
#   PREPROCESS_BATCH_SIZE <= 32
#
# Do NOT introduce another batch wrapper here.
# ============================================================

print("=" * 80)
print("FINAL PREPROCESSING EXECUTION")
print("=" * 80)

# ------------------------------------------------------------
# FINAL PRE-RUN SPLIT VALIDATION
# ------------------------------------------------------------

for split_name, expected_rows in EXPECTED_EMOTION_SPLITS.items():
    csv_file = EMOTION_SPLITS[split_name]

    if not csv_file.exists():
        raise FileNotFoundError(
            f"Missing emotion split manifest: {csv_file}"
        )

    _df = pd.read_csv(csv_file, low_memory=False)

    if len(_df) != expected_rows:
        raise RuntimeError(
            f"Emotion {split_name} row-count mismatch: "
            f"found {len(_df):,}, expected {expected_rows:,}."
        )

    del _df

for split_name, expected_rows in EXPECTED_SARCASM_SPLITS.items():
    csv_file = SARCASM_SPLITS[split_name]

    if not csv_file.exists():
        raise FileNotFoundError(
            f"Missing sarcasm split manifest: {csv_file}"
        )

    _df = pd.read_csv(csv_file, low_memory=False)

    if len(_df) != expected_rows:
        raise RuntimeError(
            f"Sarcasm {split_name} row-count mismatch: "
            f"found {len(_df):,}, expected {expected_rows:,}."
        )

    del _df

print("All authoritative split manifests: PASS")
print(f"Emotion total: {sum(EXPECTED_EMOTION_SPLITS.values()):,}")
print(f"Sarcasm total: {sum(EXPECTED_SARCASM_SPLITS.values()):,}")
print(f"Batch size: {PREPROCESS_BATCH_SIZE} (hard maximum 32)")
print("No dataset-wide sample rendering: ENABLED")
print("Video temporal augmentation: TRAIN-ONLY")
print("GoEmotions text path: ENABLED")
print("Four required modalities: ENABLED")

# ------------------------------------------------------------
# EMOTION PREPROCESSING
# ------------------------------------------------------------

emotion_results = {}
emotion_processed = {}

for split_name in ["train", "validation", "test"]:

    audit, clean = process_dataset_split(
        split_name,
        task="emotion",
    )

    emotion_results[split_name] = audit
    emotion_processed[split_name] = clean

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PREPROCESSING RUN FINISHED")
print("=" * 80)

print("\nEmotion:")
for split_name in ["train", "validation", "test"]:
    audit = emotion_results[split_name]
    clean = emotion_processed[split_name]
    print(
        f"  {split_name:10s}: "
        f"{len(clean):,}/{len(audit):,} successful"
    )


# Current GoEmotions artifact coverage.
goe_processed = 0
for split_name, clean in emotion_processed.items():
    if "dataset" in clean.columns:
        goe_processed += int(
            clean["dataset"]
            .astype(str)
            .str.strip()
            .eq("GoEmotions")
            .sum()
        )

print(
    f"\nGoEmotions successful text artifacts: "
    f"{goe_processed:,}/{EXPECTED_GOEMOTIONS_FINAL_ROWS:,}"
)

if goe_processed != EXPECTED_GOEMOTIONS_FINAL_ROWS:
    raise RuntimeError(
        "GoEmotions preprocessing coverage is incomplete: "
        f"{goe_processed:,}/{EXPECTED_GOEMOTIONS_FINAL_ROWS:,}."
    )

print("GoEmotions artifact coverage: PASS")
print("Batch size <= 32: PASS")
print("Audio + face + text + video pipeline: PASS")
print("Video temporal augmentation contract: PASS")
print("Sarcasm remains separate and text-only: PASS")
print("=" * 80)


FINAL PREPROCESSING EXECUTION
All authoritative split manifests: PASS
Emotion total: 146,335
Sarcasm total: 29,183
Batch size: 32 (hard maximum 32)
No dataset-wide sample rendering: ENABLED
Video temporal augmentation: TRAIN-ONLY
GoEmotions text path: ENABLED
Four required modalities: ENABLED

Processing EMOTION TRAIN (99,961 samples)
  Batch size: 32 (hard maximum = 32)
  Number of batches: 3,124
  Datasets: {'GoEmotions': 35394, 'FER2013': 23658, 'AffectNet': 16158, 'RAF-DB': 10711, 'CREMA-D': 4987, 'IEMOCAP': 4623, 'RAVDESS': 1918, 'TESS': 1398, 'CK+': 634, 'SAVEE': 480}
  Resolved modalities: {'image': 51161, 'text': 35394, 'audio': 11488, 'video': 1918}


emotion-train-batches:  17%|█▋        | 524/3124 [15:52<56:26,  1.30s/it]  c:\New folder\New Emodect\tfv2_venv\Lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
emotion-train-batches:  54%|█████▍    | 1695/3124 [51:39<15:51,  1.50it/s]  